In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5)

print(qc)

     
q_0: 
     
q_1: 
     
q_2: 
     
q_3: 
     
q_4: 
     


In [2]:
qc.cx(0, 3)   # A -> Difference
qc.cx(1, 3)   # B -> Difference
qc.cx(2, 3)   # Bin -> Difference

print(qc)

                    
q_0: ──■────────────
       │            
q_1: ──┼────■───────
       │    │       
q_2: ──┼────┼────■──
     ┌─┴─┐┌─┴─┐┌─┴─┐
q_3: ┤ X ├┤ X ├┤ X ├
     └───┘└───┘└───┘
q_4: ───────────────
                    


In [7]:
# Temporarily convert A to ~A
qc.x(0)

# Borrow terms
qc.ccx(0, 1, 4)   # ~A AND B
qc.ccx(0, 2, 4)   # ~A AND Bin

# Restore original A
qc.x(0)

# B AND Bin
qc.ccx(1, 2, 4)

print(qc)

          ┌───┐               ┌───┐┌───┐          ┌───┐
q_0: ──■──┤ X ├───────■────■──┤ X ├┤ X ├──■────■──┤ X ├
       │  └───┘       │    │  └───┘└───┘  │    │  └───┘
q_1: ──┼────■─────────■────┼────■─────────■────┼────■──
       │    │         │    │    │         │    │    │  
q_2: ──┼────┼────■────┼────■────■─────────┼────■────■──
     ┌─┴─┐┌─┴─┐┌─┴─┐  │    │    │         │    │    │  
q_3: ┤ X ├┤ X ├┤ X ├──┼────┼────┼─────────┼────┼────┼──
     └───┘└───┘└───┘┌─┴─┐┌─┴─┐┌─┴─┐     ┌─┴─┐┌─┴─┐┌─┴─┐
q_4: ───────────────┤ X ├┤ X ├┤ X ├─────┤ X ├┤ X ├┤ X ├
                    └───┘└───┘└───┘     └───┘└───┘└───┘


In [8]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)

# Input: A=0, B=1, Bin=0
qc.x(1)

# Difference
qc.cx(0, 3)
qc.cx(1, 3)
qc.cx(2, 3)

# Borrow
qc.x(0)

qc.ccx(0, 1, 4)
qc.ccx(0, 2, 4)

qc.x(0)

qc.ccx(1, 2, 4)

# Measurement
qc.measure([0, 1, 2, 3, 4], [0, 1, 2, 3, 4])

print(qc)

               ┌───┐               ┌───┐     ┌─┐         
q_0: ───────■──┤ X ├───────■────■──┤ X ├─────┤M├─────────
     ┌───┐  │  └───┘       │    │  └───┘     └╥┘┌─┐      
q_1: ┤ X ├──┼────■─────────■────┼─────────■───╫─┤M├──────
     └───┘  │    │         │    │         │   ║ └╥┘┌─┐   
q_2: ───────┼────┼────■────┼────■─────────■───╫──╫─┤M├───
          ┌─┴─┐┌─┴─┐┌─┴─┐  │    │   ┌─┐   │   ║  ║ └╥┘   
q_3: ─────┤ X ├┤ X ├┤ X ├──┼────┼───┤M├───┼───╫──╫──╫────
          └───┘└───┘└───┘┌─┴─┐┌─┴─┐ └╥┘ ┌─┴─┐ ║  ║  ║ ┌─┐
q_4: ────────────────────┤ X ├┤ X ├──╫──┤ X ├─╫──╫──╫─┤M├
                         └───┘└───┘  ║  └───┘ ║  ║  ║ └╥┘
c: 5/════════════════════════════════╩════════╩══╩══╩══╩═
                                     3        0  1  2  4 


In [9]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()

result = simulator.run(qc, shots=1000).result()

counts = result.get_counts()

print(counts)

{'11010': 1000}


In [10]:
def full_subtractor_circuit(a, b, bin_value):

    qc = QuantumCircuit(5, 5)

    # Prepare A
    if a == 1:
        qc.x(0)

    # Prepare B
    if b == 1:
        qc.x(1)

    # Prepare Borrow-in
    if bin_value == 1:
        qc.x(2)

    # Difference = A XOR B XOR Bin
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)

    # Borrow = (~A.B) XOR (~A.Bin) XOR (B.Bin)

    # Temporarily create ~A
    qc.x(0)

    qc.ccx(0, 1, 4)
    qc.ccx(0, 2, 4)

    # Restore A
    qc.x(0)

    # B AND Bin
    qc.ccx(1, 2, 4)

    # Measure
    qc.measure([0, 1, 2, 3, 4],
               [0, 1, 2, 3, 4])

    return qc

In [11]:
test_cases = [
    (0, 0, 0),
    (0, 0, 1),
    (0, 1, 0),
    (0, 1, 1),
    (1, 0, 0),
    (1, 0, 1),
    (1, 1, 0),
    (1, 1, 1)
]

print("A B Bin | Difference Borrow")
print("---------------------------")

for a, b, bin_value in test_cases:

    qc = full_subtractor_circuit(a, b, bin_value)

    result = simulator.run(qc, shots=100).result()

    counts = result.get_counts()

    measured = list(counts.keys())[0]

    # Qiskit order: q4 q3 q2 q1 q0
    borrow = int(measured[0])
    difference = int(measured[1])

    print(
        f"{a} {b}  {bin_value}  |"
        f"     {difference}        {borrow}"
    )

A B Bin | Difference Borrow
---------------------------
0 0  0  |     0        0
0 0  1  |     1        1
0 1  0  |     1        1
0 1  1  |     0        1
1 0  0  |     1        0
1 0  1  |     0        0
1 1  0  |     0        0
1 1  1  |     1        1


In [12]:
qc = full_subtractor_circuit(1, 0, 1)

print("Number of qubits:", qc.num_qubits)
print("Number of classical bits:", qc.num_clbits)
print("Circuit depth:", qc.depth())
print("Gate count:", qc.count_ops())

Number of qubits: 5
Number of classical bits: 5
Circuit depth: 7
Gate count: OrderedDict({'measure': 5, 'x': 4, 'cx': 3, 'ccx': 3})


In [13]:
fig = qc.draw("mpl", fold=-1)

fig.savefig(
    "../results/circuits/full_subtractor.png",
    dpi=300,
    bbox_inches="tight"
)